## Resumen
Este notebook extiende la replica del algoritmo con un modulo de outliers por peso (duplicate_households), para evaluar si la duplicacion/reasignacion de pesos mejora la monotonicidad de welfare_share y su impacto en la media y en el top tail.

1. Setup

In [27]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

2. Paths

In [ ]:
ROOT = Path(".").resolve()
INPUT = ROOT / "01-input"
OUT = ROOT / "work"

mwi_path = INPUT / "country" / "MWI_1997.dta"
pip_bins_path = INPUT / "pip_1kbins.dta"
global_bins_path = INPUT / "GlobalDist1000bins_1990_2026_20260324_2021_01_02_PROD.dta"

mwi_path, pip_bins_path, global_bins_path

(PosixPath('/Users/danielaavayu/Desktop/World Bank/Bottom Coding/01-input/country/MWI_1997.dta'),
 PosixPath('/Users/danielaavayu/Desktop/World Bank/Bottom Coding/01-input/pip_1kbins.dta'),
 PosixPath('/Users/danielaavayu/Desktop/World Bank/Bottom Coding/01-input/GlobalDist1000bins_1990_2026_20260324_2021_01_02_PROD.dta'))

Chunk 3: Load Benchmarks

In [29]:
pip_bins = pd.read_stata(pip_bins_path, convert_categoricals=False)
pip_bins["year"] = pip_bins["year"].astype(int)

mwi_pip_row = pip_bins[
    (pip_bins["country_code"] == "MWI")
    & (pip_bins["year"] == 1997)
].iloc[0]

pip_mean = mwi_pip_row["mean"]
global_1kbins_mean = mwi_pip_row["mean_1kbins"]

pd.DataFrame({
    "source": ["PIP mean", "Global 1000-bin mean"],
    "mean": [pip_mean, global_1kbins_mean],
})

,source,mean
0,PIP mean,5.724018
1,Global 1000-bin mean,4.865801


Chunk 4: Load Malawi Microdata

In [30]:
mwi_raw = pd.read_stata(mwi_path, convert_categoricals=False)

cpi2021 = mwi_raw["cpi2021"].dropna().iloc[0]
icp2021 = mwi_raw["icp2021"].dropna().iloc[0]

mwi = mwi_raw.copy()

mwi["welfare_2021ppp_day"] = (
    mwi["welfare"] / (cpi2021 * icp2021 * 365)
)

mwi = mwi[["welfare", "welfare_2021ppp_day", "weight"]].dropna().copy()
mwi = mwi[mwi["weight"] > 0].copy()

# The R code expects these names.
mwi["welfare"] = mwi["welfare_2021ppp_day"]

# If reporting_level does not exist in the raw country file, create national.
mwi["reporting_level"] = "national"

mwi[["welfare", "weight"]].describe()

,welfare,weight
count,"10,698.000000","10,698.000000"
mean,10.553808,903.593933
std,529.936157,608.497070
min,0.202600,59.504238
25%,1.799609,459.908325
50%,2.723850,778.125000
75%,4.382512,"1,185.886475"
max,"53,435.875000","6,765.000000"


Chunk 5: Load Delivered Global Bins For Comparison

In [31]:
chunks = []

for chunk in pd.read_stata(
    global_bins_path,
    columns=["code", "year", "quantile", "welf", "pop"],
    chunksize=500_000,
):
    chunk["year"] = chunk["year"].astype(int)

    keep = (
        (chunk["code"] == "MWI")
        & (chunk["year"] == 1997)
    )

    if keep.any():
        chunks.append(chunk.loc[keep].copy())

mwi_global_bins = pd.concat(chunks, ignore_index=True)
mwi_global_bins["quantile"] = mwi_global_bins["quantile"].astype(float)
mwi_global_bins = mwi_global_bins.sort_values("quantile").reset_index(drop=True)

mwi_global_bins.tail()

,code,year,quantile,welf,pop
995,MWI,1997,996.000000,46.885646,0.010562
996,MWI,1997,997.000000,56.900297,0.010562
997,MWI,1997,998.000000,70.545418,0.010562
998,MWI,1997,999.000000,82.738549,0.010562
999,MWI,1997,"1,000.000000","1,296.486318",0.010562


6: Helper Functions From The R Algorithm (Andres)

In [32]:
def find_outliers_by_weight(df, weight_col="weight", threshold=2.5):
    out = df.copy()
    mean_w = out[weight_col].mean()
    sd_w = out[weight_col].std(ddof=1)

    out["is_outlier"] = out[weight_col] > (mean_w + threshold * sd_w)
    return out


def optimize_ratio(y, m):
    y = np.asarray(y, dtype=float)
    opt_x = np.maximum(m, np.sqrt(y))

    to_one = opt_x > (y / 2)
    opt_x[to_one] = 1

    return np.round(y / opt_x).astype(int)


def duplicate_obs(df, weight_col="weight"):
    out = df.copy()
    min_w = out[weight_col].min()

    out["hhindex"] = np.arange(len(out))
    out["rep_count"] = optimize_ratio(out[weight_col].to_numpy(), min_w)

    out.loc[~out["is_outlier"], "rep_count"] = 1
    out["rep_count"] = out["rep_count"].astype(int)

    duplicated = out.loc[out.index.repeat(out["rep_count"])].copy()
    return duplicated


def add_new_weights(df, weight_col="weight"):
    out = df.copy()

    # For rows originally flagged as weight outliers, split weight across replications.
    out.loc[out["is_outlier"], weight_col] = (
        out.loc[out["is_outlier"], weight_col]
        / out.loc[out["is_outlier"], "rep_count"]
    )

    return out

7: Implement The Delivered new_bins Logic

In [33]:
def new_bins_delivered(welfare, weight, nbins=1000, tolerance=1e-6, ids=None):
    welfare = np.asarray(welfare, dtype=float)
    weight = np.asarray(weight, dtype=float)

    valid = ~np.isnan(welfare) & ~np.isnan(weight)
    welfare = welfare[valid]
    weight = weight[valid]

    if ids is None:
        ids = np.arange(len(welfare))
    else:
        ids = np.asarray(ids)[valid]

    order = np.argsort(welfare)
    welfare = welfare[order]
    weight = weight[order]
    ids = ids[order]

    total_weight = weight.sum()
    bin_size = total_weight / nbins

    out_rows = []

    cur_bin = 1
    cur_weight = 0.0

    for id_i, w, wt in zip(ids, welfare, weight):
        while wt > 0 and cur_bin <= nbins:
            room = bin_size - cur_weight

            if abs(room) < tolerance:
                cur_bin += 1
                cur_weight = 0.0
                continue

            take = min(wt, room)

            out_rows.append({
                "id": id_i,
                "bin": cur_bin,
                "weight": take,
                "welfare": w,
            })

            wt -= take
            cur_weight += take

            if cur_weight >= bin_size - tolerance:
                cur_bin += 1
                cur_weight = 0.0

    return pd.DataFrame(out_rows)

8: Implement lorenz_table Like The R Code

In [34]:
def lorenz_table_delivered(df, nq=1000, tolerance=1e-6):
    d = df.copy()

    if "reporting_level" not in d.columns:
        d["reporting_level"] = "national"

    pieces = []

    for reporting_level, sub in d.groupby("reporting_level", observed=True):
        sub = sub.sort_values("welfare").reset_index(drop=True)
        sub["id"] = np.arange(len(sub))

        bins = new_bins_delivered(
            welfare=sub["welfare"].to_numpy(),
            weight=sub["weight"].to_numpy(),
            ids=sub["id"].to_numpy(),
            nbins=nq,
            tolerance=tolerance,
        )

        bins["reporting_level"] = reporting_level
        pieces.append(bins)

    expanded = pd.concat(pieces, ignore_index=True)

    expanded["wt_welfare"] = expanded["welfare"] * expanded["weight"]

    totals = (
        expanded.groupby("reporting_level", observed=True)
        .agg(
            tot_pop=("weight", "sum"),
            tot_wlf=("wt_welfare", "sum"),
        )
        .reset_index()
    )

    expanded = expanded.merge(totals, on="reporting_level", how="left")

    expanded["pop_share"] = expanded["weight"] / expanded["tot_pop"]
    expanded["welfare_share"] = expanded["wt_welfare"] / expanded["tot_wlf"]

    lt = (
        expanded.groupby(["reporting_level", "bin"], observed=True)
        .agg(
            avg_welfare=("welfare", lambda x: np.average(
                x,
                weights=expanded.loc[x.index, "weight"],
            )),
            pop_share=("pop_share", "sum"),
            welfare_share=("welfare_share", "sum"),
            quantile=("welfare", "max"),
            pop=("weight", "sum"),
        )
        .reset_index()
        .sort_values(["reporting_level", "bin"])
    )

    return lt

9: Implement The Household Duplication Wrapper

In [35]:
def duplicate_households_delivered(
    df,
    weight_col="weight",
    threshold=2.5,
    i=0,
    li=5,
    super_limit=20,
    nq=1000,
):
    R = df.copy()

    # First check: do bins already satisfy monotonic welfare_share?
    if i == 0:
        lt = lorenz_table_delivered(R, nq=nq)
        welfare_share_bad = (
            lt.groupby("reporting_level", observed=True)["welfare_share"]
            .apply(lambda s: (s.diff().dropna() < 0).any())
            .any()
        )

        if not welfare_share_bad:
            return R, lt, {
                "welfare_share_OK": True,
                "threshold": threshold,
                "iterations": i,
            }

    if i >= super_limit:
        lt = lorenz_table_delivered(R, nq=nq)
        return R, lt, {
            "welfare_share_OK": False,
            "threshold": threshold,
            "iterations": i,
        }

    while True:
        i += 1

        R = find_outliers_by_weight(R, weight_col=weight_col, threshold=threshold)
        R = duplicate_obs(R, weight_col=weight_col)
        R = add_new_weights(R, weight_col=weight_col)

        # Drop helper columns, as in clean_new_weights().
        helper_cols = ["is_outlier", "hhindex", "rep_count"]
        R = R.drop(columns=[c for c in helper_cols if c in R.columns])

        lt = lorenz_table_delivered(R, nq=nq)

        welfare_share_bad = (
            lt.groupby("reporting_level", observed=True)["welfare_share"]
            .apply(lambda s: (s.diff().dropna() < 0).any())
            .any()
        )

        if not welfare_share_bad:
            return R, lt, {
                "welfare_share_OK": True,
                "threshold": threshold,
                "iterations": i,
            }

        if i >= super_limit:
            return R, lt, {
                "welfare_share_OK": False,
                "threshold": threshold,
                "iterations": i,
            }

        if i >= li and threshold > 0:
            threshold = max(threshold - 0.5, 0)
            li = li * 2

10: Run The Delivered Algorithm For Malawi 1997

In [36]:
mwi_for_algorithm = mwi[["welfare", "weight", "reporting_level"]].copy()

mwi_replicated, lt_delivered_style, algorithm_info = duplicate_households_delivered(
    mwi_for_algorithm,
    weight_col="weight",
    threshold=2.5,
    i=0,
    li=5,
    super_limit=20,
    nq=1000,
)

algorithm_info

{'welfare_share_OK': True, 'threshold': 2.5, 'iterations': 0}

11: Compare Row Counts And Weight Totals

In [37]:
pd.DataFrame({
    "object": [
        "Original microdata",
        "After delivered-style household duplication",
    ],
    "n_rows": [
        len(mwi_for_algorithm),
        len(mwi_replicated),
    ],
    "total_weight": [
        mwi_for_algorithm["weight"].sum(),
        mwi_replicated["weight"].sum(),
    ],
    "weighted_mean": [
        np.average(mwi_for_algorithm["welfare"], weights=mwi_for_algorithm["weight"]),
        np.average(mwi_replicated["welfare"], weights=mwi_replicated["weight"]),
    ],
})

,object,n_rows,total_weight,weighted_mean
0,Original microdata,10698,"9,666,648.000000",5.779648
1,After delivered-style household duplication,10698,"9,666,648.000000",5.779648


12: Compute Mean From Delivered-Style Bins

In [38]:
delivered_style_mean = np.average(
    lt_delivered_style["avg_welfare"],
    weights=lt_delivered_style["pop"],
)

pd.DataFrame({
    "source": [
        "PIP mean",
        "Global 1000-bin mean from delivered file",
        "Mean from delivered-style replicated bins",
        "Original microdata direct mean",
    ],
    "mean": [
        pip_mean,
        global_1kbins_mean,
        delivered_style_mean,
        np.average(mwi["welfare"], weights=mwi["weight"]),
    ],
})

,source,mean
0,PIP mean,5.724018
1,Global 1000-bin mean from delivered file,4.865801
2,Mean from delivered-style replicated bins,5.779648
3,Original microdata direct mean,5.779648


13: Compare Delivered-Style Bins To Global Bins

In [39]:
lt_compare = lt_delivered_style.rename(
    columns={
        "bin": "quantile",
        "avg_welfare": "delivered_style_bin_welf",
        "pop": "delivered_style_pop",
    }
)

lt_compare["quantile"] = lt_compare["quantile"].astype(float)

bin_comparison = lt_compare.merge(
    mwi_global_bins[["quantile", "welf", "pop"]],
    on="quantile",
    how="left",
).rename(columns={
    "welf": "global_bin_welf",
    "pop": "global_bin_pop",
})

bin_comparison["diff_delivered_style_minus_global"] = (
    bin_comparison["delivered_style_bin_welf"]
    - bin_comparison["global_bin_welf"]
)

bin_comparison["ratio_delivered_style_global"] = (
    bin_comparison["delivered_style_bin_welf"]
    / bin_comparison["global_bin_welf"]
)

bin_comparison.tail(20)

ValueError: The column label 'quantile' is not unique.

14: Check Whether You Reproduce The 4.8658 Mean

In [ ]:
check_mean = pd.DataFrame({
    "measure": [
        "Global 1k-bin mean from pip_1kbins.dta",
        "Mean from reproduced delivered-style bins",
        "Difference",
    ],
    "value": [
        global_1kbins_mean,
        delivered_style_mean,
        delivered_style_mean - global_1kbins_mean,
    ],
})

check_mean

,measure,value
0,Global 1k-bin mean from pip_1kbins.dta,4.865801
1,Mean from reproduced delivered-style bins,5.779648
2,Difference,0.913847


15: Graph Top Tail

In [ ]:
top_tail = bin_comparison[bin_comparison["quantile"] >= 950].copy()

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    top_tail["quantile"],
    top_tail["delivered_style_bin_welf"],
    label="Reproduced delivered-style bins",
    linewidth=1.5,
)

ax.plot(
    top_tail["quantile"],
    top_tail["global_bin_welf"],
    label="Global delivered bins",
    linewidth=1.5,
    linestyle="--",
)

ax.set_title("Malawi 1997: reproduced delivered-style bins vs global bins")
ax.set_xlabel("Bin / quantile")
ax.set_ylabel("Mean welfare, 2021 PPP USD per person per day")
ax.legend()

plt.tight_layout()
plt.show()